In [1]:
%pip install joblib numpy pandas scikit-learn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Train DDoS Prediction Model

This notebook trains a binary classifier from the converted CIC-DDoS2019 CSV files. The saved model predicts whether a network flow is `Benign` or `DDoS`.

In [2]:
# Install and import required libraries.
import importlib
import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path

REQUIRED_PACKAGES = {
    "joblib": "joblib",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}

for import_name, package_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

def progress(message):
    print(f"[progress] {message}")

progress("Setup complete.")

[progress] Setup complete.


In [3]:
# Resolve project paths.
cwd = Path.cwd().resolve()
training_dir = cwd if cwd.name.lower() == "training" else cwd / "training"
csv_dir = training_dir / "dataset" / "csv"
model_dir = training_dir / "model"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "ddos_binary_classifier.joblib"
metadata_path = model_dir / "ddos_binary_classifier_metadata.json"

print(f"CSV directory: {csv_dir}")
print(f"Model output: {model_path}")

if not csv_dir.exists():
    raise FileNotFoundError(f"CSV directory not found: {csv_dir}")

csv_files = sorted(csv_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files found in: {csv_dir}")

print(f"CSV files found: {len(csv_files)}")

CSV directory: C:\Users\perso\EJIE\Work Related\Detect Adaptive Difficulty Scoring DDoS Management System\training\dataset\csv
Model output: C:\Users\perso\EJIE\Work Related\Detect Adaptive Difficulty Scoring DDoS Management System\training\model\ddos_binary_classifier.joblib
CSV files found: 17


In [4]:
# Load and combine the CSV files.
progress("Loading CSV files...")
frames = []

for csv_file in csv_files:
    df_part = pd.read_csv(csv_file)
    df_part["Source File"] = csv_file.name
    frames.append(df_part)
    print(f"Loaded {csv_file.name}: {len(df_part):,} rows")

df = pd.concat(frames, ignore_index=True)
before_drop = len(df)
df = df.drop_duplicates().reset_index(drop=True)
dropped = before_drop - len(df)

print(f"Combined rows: {before_drop:,}")
print(f"Rows after duplicate removal: {len(df):,}")
print(f"Duplicates removed: {dropped:,}")
print(f"Columns: {len(df.columns)}")
progress("CSV loading complete.")

[progress] Loading CSV files...
Loaded DNS-testing.csv: 6,703 rows
Loaded LDAP-testing.csv: 2,831 rows
Loaded LDAP-training.csv: 6,715 rows
Loaded MSSQL-testing.csv: 8,083 rows
Loaded MSSQL-training.csv: 10,974 rows
Loaded NetBIOS-testing.csv: 2,225 rows
Loaded NetBIOS-training.csv: 1,631 rows
Loaded NTP-testing.csv: 134,674 rows
Loaded Portmap-training.csv: 5,105 rows
Loaded SNMP-testing.csv: 4,018 rows
Loaded Syn-testing.csv: 907 rows
Loaded Syn-training.csv: 70,336 rows
Loaded TFTP-testing.csv: 121,833 rows
Loaded UDP-testing.csv: 12,462 rows
Loaded UDP-training.csv: 17,770 rows
Loaded UDPLag-testing.csv: 12,465 rows
Loaded UDPLag-training.csv: 12,639 rows
Combined rows: 431,371
Rows after duplicate removal: 431,368
Duplicates removed: 3
Columns: 79
[progress] CSV loading complete.


In [5]:
# Prepare features and binary labels.
target_column = "Label"
drop_columns = [target_column, "Source File"]

if target_column not in df.columns:
    raise ValueError(f"Expected target column '{target_column}' was not found.")

feature_columns = [column for column in df.columns if column not in drop_columns]
X = df[feature_columns].replace([np.inf, -np.inf], np.nan)
y = (df[target_column].astype(str).str.lower() != "benign").astype(int)

label_mapping = {"Benign": 0, "DDoS": 1}

print(f"Feature columns: {len(feature_columns)}")
print("Binary label distribution:")
print(y.map({0: "Benign", 1: "DDoS"}).value_counts())
print("Original attack labels:")
print(df[target_column].value_counts().sort_index())

Feature columns: 77
Binary label distribution:
Label
DDoS      333540
Benign     97828
Name: count, dtype: int64
Original attack labels:
Label
Benign            97828
DrDoS_DNS          3669
DrDoS_LDAP         1440
DrDoS_MSSQL        6212
DrDoS_NTP        121368
DrDoS_NetBIOS       598
DrDoS_SNMP         2717
DrDoS_UDP         10420
LDAP               1906
MSSQL              8523
NetBIOS             644
Portmap             685
Syn               49373
TFTP              98917
UDP               18090
UDP-lag            8872
UDPLag               55
WebDDoS              51
Name: count, dtype: int64


In [6]:
# Train/evaluate with a stratified holdout split.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            HistGradientBoostingClassifier(
                max_iter=200,
                learning_rate=0.08,
                max_leaf_nodes=31,
                l2_regularization=0.1,
                random_state=42,
            ),
        ),
    ]
)

progress("Training evaluation model...")
pipeline.fit(X_train, y_train)
progress("Evaluation model training complete.")

[progress] Training evaluation model...
[progress] Evaluation model training complete.


In [7]:
# Evaluate model performance.
y_pred = pipeline.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
report = classification_report(
    y_test,
    y_pred,
    target_names=["Benign", "DDoS"],
    output_dict=True,
)
matrix = confusion_matrix(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 score: {f1:.4f}")
print("Confusion matrix [[Benign->Benign, Benign->DDoS], [DDoS->Benign, DDoS->DDoS]]:")
print(matrix)
print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["Benign", "DDoS"]))

Accuracy: 0.9996
F1 score: 0.9997
Confusion matrix [[Benign->Benign, Benign->DDoS], [DDoS->Benign, DDoS->DDoS]]:
[[19552    14]
 [   24 66684]]
Classification report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00     19566
        DDoS       1.00      1.00      1.00     66708

    accuracy                           1.00     86274
   macro avg       1.00      1.00      1.00     86274
weighted avg       1.00      1.00      1.00     86274



In [8]:
# Train final model on all available rows and save artifacts.
progress("Training final model on all rows...")
final_model = clone(pipeline)
final_model.fit(X, y)
progress("Final model training complete.")

joblib.dump(final_model, model_path)

metadata = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "model_type": "HistGradientBoostingClassifier binary DDoS classifier",
    "model_path": str(model_path),
    "training_rows": int(len(df)),
    "feature_count": int(len(feature_columns)),
    "feature_columns": feature_columns,
    "label_mapping": label_mapping,
    "source_csv_files": [file.name for file in csv_files],
    "holdout_metrics": {
        "accuracy": float(accuracy),
        "f1_score": float(f1),
        "confusion_matrix": matrix.tolist(),
        "classification_report": report,
    },
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved model: {model_path}")
print(f"Saved metadata: {metadata_path}")

[progress] Training final model on all rows...
[progress] Final model training complete.
Saved model: C:\Users\perso\EJIE\Work Related\Detect Adaptive Difficulty Scoring DDoS Management System\training\model\ddos_binary_classifier.joblib
Saved metadata: C:\Users\perso\EJIE\Work Related\Detect Adaptive Difficulty Scoring DDoS Management System\training\model\ddos_binary_classifier_metadata.json


In [9]:
# Quick load test using one sample row.
loaded_model = joblib.load(model_path)
sample_row = X.iloc[[0]]
prediction = loaded_model.predict(sample_row)[0]
probability = loaded_model.predict_proba(sample_row)[0]
prediction_label = "DDoS" if prediction == 1 else "Benign"

print(f"Prediction: {prediction_label}")
print(f"Probability [Benign, DDoS]: {probability}")
progress("Model load test complete.")

Prediction: DDoS
Probability [Benign, DDoS]: [9.01760075e-06 9.99990982e-01]
[progress] Model load test complete.


In [10]:
# Simulate a few real traffic samples and check whether the model gets them right.
simulation_model = joblib.load(model_path)

simulation_size = min(8, len(X_test))
simulation_indices = X_test.sample(n=simulation_size, random_state=42).index
simulation_X = X_test.loc[simulation_indices]
simulation_y = y_test.loc[simulation_indices]

simulation_predictions = simulation_model.predict(simulation_X)
simulation_probabilities = simulation_model.predict_proba(simulation_X)

simulation_results = pd.DataFrame({
    "Actual": simulation_y.map({0: "Benign", 1: "DDoS"}).values,
    "Predicted": pd.Series(simulation_predictions).map({0: "Benign", 1: "DDoS"}).values,
    "Benign_Prob": simulation_probabilities[:, 0],
    "DDoS_Prob": simulation_probabilities[:, 1],
}, index=simulation_X.index)

print("Simulation results on a few holdout rows:")
print(simulation_results.to_string())

simulation_accuracy = (simulation_results["Actual"] == simulation_results["Predicted"]).mean()
print(f"\nSimulation accuracy: {simulation_accuracy:.3f}")

benign_example = simulation_results[simulation_results["Actual"] == "Benign"].head(1)
ddos_example = simulation_results[simulation_results["Actual"] == "DDoS"].head(1)

if not benign_example.empty:
    print("\nExample benign flow:")
    print(benign_example.to_string())

if not ddos_example.empty:
    print("\nExample DDoS flow:")
    print(ddos_example.to_string())

progress("Simulation sanity check complete.")


Simulation results on a few holdout rows:
        Actual Predicted   Benign_Prob  DDoS_Prob
344275    DDoS      DDoS  6.003911e-07   0.999999
428994  Benign    Benign  9.999814e-01   0.000019
74672     DDoS      DDoS  4.573617e-07   1.000000
123237    DDoS      DDoS  5.936366e-07   0.999999
181998  Benign    Benign  9.999949e-01   0.000005
159198    DDoS      DDoS  3.601987e-07   1.000000
116942    DDoS      DDoS  2.860308e-07   1.000000
314220    DDoS      DDoS  5.774050e-07   0.999999

Simulation accuracy: 1.000

Example benign flow:
        Actual Predicted  Benign_Prob  DDoS_Prob
428994  Benign    Benign     0.999981   0.000019

Example DDoS flow:
       Actual Predicted   Benign_Prob  DDoS_Prob
344275   DDoS      DDoS  6.003911e-07   0.999999
[progress] Simulation sanity check complete.


In [11]:
# Stronger holdout validation on a larger sample.
validation_model = joblib.load(model_path)

validation_size = min(500, len(X_test))
validation_X = X_test.sample(n=validation_size, random_state=123)
validation_y = y_test.loc[validation_X.index]
validation_predictions = validation_model.predict(validation_X)

validation_accuracy = accuracy_score(validation_y, validation_predictions)
validation_f1 = f1_score(validation_y, validation_predictions)
validation_matrix = confusion_matrix(validation_y, validation_predictions)
validation_report = classification_report(
    validation_y,
    validation_predictions,
    target_names=["Benign", "DDoS"],
)

print(f"Validation sample size: {validation_size}")
print(f"Validation accuracy: {validation_accuracy:.4f}")
print(f"Validation F1 score: {validation_f1:.4f}")
print("Confusion matrix [[Benign->Benign, Benign->DDoS], [DDoS->Benign, DDoS->DDoS]]:")
print(validation_matrix)
print("Classification report:")
print(validation_report)

progress("Stronger validation check complete.")


Validation sample size: 500
Validation accuracy: 1.0000
Validation F1 score: 1.0000
Confusion matrix [[Benign->Benign, Benign->DDoS], [DDoS->Benign, DDoS->DDoS]]:
[[ 94   0]
 [  0 406]]
Classification report:
              precision    recall  f1-score   support

      Benign       1.00      1.00      1.00        94
        DDoS       1.00      1.00      1.00       406

    accuracy                           1.00       500
   macro avg       1.00      1.00      1.00       500
weighted avg       1.00      1.00      1.00       500

[progress] Stronger validation check complete.


In [12]:
# Compare performance on benign-heavy and DDoS-heavy slices.
comparison_model = joblib.load(model_path)

benign_pool = y_test[y_test == 0].index
ddos_pool = y_test[y_test == 1].index

benign_size = min(100, len(benign_pool))
ddos_size = min(100, len(ddos_pool))

benign_X = X_test.loc[benign_pool].sample(n=benign_size, random_state=7)
benign_y = y_test.loc[benign_X.index]
ddos_X = X_test.loc[ddos_pool].sample(n=ddos_size, random_state=7)
ddos_y = y_test.loc[ddos_X.index]

benign_pred = comparison_model.predict(benign_X)
ddos_pred = comparison_model.predict(ddos_X)

benign_acc = accuracy_score(benign_y, benign_pred)
ddos_acc = accuracy_score(ddos_y, ddos_pred)
benign_f1 = f1_score(benign_y, benign_pred)
ddos_f1 = f1_score(ddos_y, ddos_pred)
benign_cm = confusion_matrix(benign_y, benign_pred)
ddos_cm = confusion_matrix(ddos_y, ddos_pred)

print(f"Benign-heavy sample size: {benign_size}")
print(f"Benign-heavy accuracy: {benign_acc:.4f}")
print(f"Benign-heavy F1 score: {benign_f1:.4f}")
print("Benign-heavy confusion matrix [[Benign->Benign, Benign->DDoS], [DDoS->Benign, DDoS->DDoS]]:")
print(benign_cm)

print(f"\nDDoS-heavy sample size: {ddos_size}")
print(f"DDoS-heavy accuracy: {ddos_acc:.4f}")
print(f"DDoS-heavy F1 score: {ddos_f1:.4f}")
print("DDoS-heavy confusion matrix [[Benign->Benign, Benign->DDoS], [DDoS->Benign, DDoS->DDoS]]:")
print(ddos_cm)

comparison_summary = pd.DataFrame({
    "Slice": ["Benign-heavy", "DDoS-heavy"],
    "Sample Size": [benign_size, ddos_size],
    "Accuracy": [benign_acc, ddos_acc],
    "F1 Score": [benign_f1, ddos_f1],
})

print("\nSide-by-side summary:")
print(comparison_summary.to_string(index=False))

progress("Class balance comparison complete.")


Benign-heavy sample size: 100
Benign-heavy accuracy: 1.0000
Benign-heavy F1 score: 0.0000
Benign-heavy confusion matrix [[Benign->Benign, Benign->DDoS], [DDoS->Benign, DDoS->DDoS]]:
[[100]]

DDoS-heavy sample size: 100
DDoS-heavy accuracy: 1.0000
DDoS-heavy F1 score: 1.0000
DDoS-heavy confusion matrix [[Benign->Benign, Benign->DDoS], [DDoS->Benign, DDoS->DDoS]]:
[[100]]

Side-by-side summary:
       Slice  Sample Size  Accuracy  F1 Score
Benign-heavy          100       1.0       0.0
  DDoS-heavy          100       1.0       1.0
[progress] Class balance comparison complete.


c:\Users\perso\anaconda3\envs\ejie_dev3.10\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\perso\anaconda3\envs\ejie_dev3.10\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
c:\Users\perso\anaconda3\envs\ejie_dev3.10\lib\site-packages\sklearn\metrics\_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
